# Section 1 - Quantifying the association of a feature with an outcome

* We frequently want to know if a **feature** is **associated** with an **outcome**. For example, we might want to determine:
    * Is the presence of a specific cell stress condition (feature) associated with the formation of stress granules (outcome)?
    * Is the amino acid sequence of a protein (feature) associated with its propensity to misfold (outcome)?
* This type of analysis is invaluable for checking for correlations between features and outcomes in complex data
* In this section we will explore using contingency tables, odds ratios, and Fisher's Exact Test to test for association

---

### Before you begin
* Remember - [download the background & theory packet](https://drive.google.com/file/d/1sOxAyo2lRvEjULoF53rvWa9uz6nlysDN/view?usp=drive_link) accompanying this training 

---

## Example 1.1

**Application 1.1**: We hypothesize that *E. coli* proteins that contain native entanglements are more likely to misfold than proteins without entanglements.

* To proceed, we need information about which proteins in *E. coli* contain entanglements and which proteins in *E. coli* misfold. 

### What is an entanglement? How can we tell if a protein is entangled?

* Entanglements are a structural motif in proteins formed by two segments: a loop (closed by a native contact) and a thread (**Figures 1.1.1** & **1.1.2**).

![](../images/entanglement-2D.png)

**Figure 1.1.1**. *General structure of a non-covalent lasso entanglement. The threading segment (blue) passes through a loop (red) that is closed by a native contact (yellow). J. Mol. Biol. 436 (2024) 168487.*

![](../images/native-and-entangled-states.png)

**Figure 1.1.2**. *3D structures of oligoribonuclease without (left) and with (right) an entanglement. The threading segment (blue) passes through a loop (red) that is closed by a native contact (yellow). J. Mol. Biol. 436 (2024) 168487.*

* Some proteins contain entanglements in their native state (*i.e.*, native entanglements)
  
* Some proteins can gain or lose entanglements during misfolding

* For the current hypothesis, we are concerned with native entanglements; this information can be obtained by analyzing either experimental structures or predicted structures

* Our analysis will use data on entanglements computed from experimental structures of *E. coli* proteins

### How can we tell if a protein misfolds?

* The structural proteomics technique *limited proteolysis mass spectrometry* (LiP-MS) profiles changes in protein structures across the proteome in response to perturbations (**Figure 1.1.3**)

![](../images/lip-ms.png)

**Figure 1.1.3**. *Schematic of a LiP-MS experiment. When studying misfolding, one sample will be treated with guanidinium chloride to induce unfolding before a dilution jump is used to stimulate refolding; the other sample is not treated with guanidinium chloride, preserving protein native states. Nature Protocols volume 12, p. 2391–2410 (2017).*

* LiP-MS compares differences in protein structures between two samples

* In the case at hand, a protein is considered to misfold if there is a significant change in its limited proteolysis digestion pattern between a guanidinium chloride-unfolded/refolded and an untreated sample containing natively folded protein

* We will use LiP-MS data from *E. coli* to match the *E. coli* entanglement data

### We have our data - what now?

* Now that we have identified relevant data to test our hypothesis, let's dive into some code. 

## Testing our hypothesis in Python

### Step 0 - Load libraries

* We first need to make sure we have access to all of the functions etc. that we need for this analysis - let's load some libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

### Step 1 - Load the data
* After loading the libraries, we now need to load the data into memory

In [ ]:
# "data1" is a pandas DataFrame object
data_path = "/home/jovyan/data-store/home/shared/NCEMS/2025-NCEMS-ML-webinar/"
data1     = pd.read_csv(data_path + "NativeEntanglements_and_SigCuts_EXP_buffC.csv")

### Step 2 - Explore the data

* Let's explore the data quickly to get a better understanding of what we need to do

In [ ]:
# first, print a quick summary
print ("Create a quick summary of the DataFrame:\n")
data1.info()

# second, print the first 10 rows of data1
print ("\nPrint the first 10 rows of the DataFrame:\n")
display(data1.head(10))

# third, count the number of unique gene identifiers in column "gene" of data1
print ("\nThe number of unique genes is:", len(data1["gene"].unique()))

* We can see from this summary that this data file contains 4 columns:
    * `buff`: the buffer condition for the experiment
    * `gene`: the unique gene identifier (there are no duplicates!)
    * `NativeEnt`: `True` if the protein has a native entanglement, `False` if it does not
    * `NonRefoldable`: `True` if the protein *did not* refold in LiP-MS experiment (i.e., misfolded), `False` if it *did* refold
<pre>

</pre>
* Now that we have a better understanding of the data, we are ready to run our analysis.

### Step 3 - Run the analysis

In [ ]:
# compute the values of {a, b, c, d} and construct the contingency table
a = len(data1[(data1["NativeEnt"] == True ) & (data1["NonRefoldable"] == True )])
b = len(data1[(data1["NativeEnt"] == True ) & (data1["NonRefoldable"] == False)])
c = len(data1[(data1["NativeEnt"] == False) & (data1["NonRefoldable"] == True )])
d = len(data1[(data1["NativeEnt"] == False) & (data1["NonRefoldable"] == False)])

# put values into a new format to enable a nice print statement & analysis
contingency_table = pd.DataFrame({"Protein Misfolded"    : [a, c], 
                                  "Protein Not Misfolded": [b, d]},
                                 index = ["Protein Entangled", "Protein Not Entangled"])

# print the output
print ("This is our contingency table:\n")

# create a table from our contingency_table using matplotlib
plt.clf()
fig, ax    = plt.subplots(figsize = (5, 2))
ax.axis("tight")
ax.axis("off")
cell_text  = contingency_table.reset_index().values.tolist()
col_labels = [""] + contingency_table.columns.tolist()
table      = ax.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(14)  
table.scale(2, 2)  
plt.show()

* We can complete this analysis by computing the odds ratio and *p*-value

In [ ]:
# use the fisher_exact function from scipy.stats to compute the odds ratio and p-value
odds_ratio, fisher_p_value = fisher_exact(contingency_table, alternative = "two-sided")

# print the results of this analysis
print ("The odds ratio is:", "%.2f" %odds_ratio)
print ("The p-value is   :", "%.2e" %fisher_p_value)

### Step 4 - Interpret the results

* The odds ratio of 4.19 indicates that there is a **positive association** between entanglement and misfolding
* In other words, entanglement and misfolding tend to co-occur in the same protein, supporting our hypothesis
    * Odds ratios > 1 indicate positive association
    * Odds ratios = 1 indicate no association
    * Odds ratios < 1 indicate negative association
* We can also say that the **odds** of an entangled protein misfolding are 4.19 times greater than the odds of a non-entangled protein misfolding
* The *p*-value is <<0.05, which is a common threshold for significance; in this instance, we conclude the result is significant

---

## Example 1.2

**Application 1.2**: We hypothesize that, because proteins containing native entanglements are more likely to misfold, they are also more likely to be linked with disease than proteins lacking native entanglements.

* One way to test this hypothesis is to once again create a 2 x 2 contingency table, compute the odds ratio, and then use Fisher's Exact Test to compute a *p*-value
* We need information on proteins that contain native entanglements as well as information on which proteins are implicated in disease

### Step 0 - Load libraries
* Run the below code cell to load all of the libraries needed for the subsequent analysis

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

### Step 1 - Load the data

* One of the provided data files includes entanglement status and the proteins linked with disease in humans in a readily usable form
* The entanglement status for human proteins was determined by analyzing AlphaFold protein structure predictions; proteins with low-quality predictions were eliminated from the analysis
* Links between proteins and disease were determined using the database DISGENET (https://disgenet.com/)

In [ ]:
# "data2" is a pandas DataFrame object
data_path = "/home/jovyan/data-store/home/shared/NCEMS/2025-NCEMS-ML-webinar/"
data2     = pd.read_csv(data_path+"entanglement-disease-association.csv")

### Step 2 - Explore the data

* We will once again run a few simple commands to explore the data

In [ ]:
# first, print a quick summary
print ("Create a quick summary of the DataFrame:\n")
data2.info()

# second, print the first 10 rows of data2
print ("\nPrint the first 10 rows of the DataFrame:\n")
display(data2.head(10))

# third, check to see if all rows correspond to a unique gene identifier
N_unique = len(data2["gene"].unique())
print ("The number of unique gene IDs is:", N_unique)

* We can see from these results that there are 5,366 rows in the table and all of the entries in all columns have values (i.e., there are no `NaN` entries)
* We can also see that there are no duplicate rows - the number of unique values in the column `gene` is equal to the number of rows in `data2`

### Step 3 - Run the analysis

* Now that we have loaded and examined our data, we are ready to carry out the analysis of the association
* Take a minute to think about what your contingency table will look like; what will the rows and columns represent? When you have your answer, run the below code cell to see a sketch of the contingency table for this hypothesis

In [ ]:
# print a blank contingency table in the format needed for this hypothesis
contingency_table = pd.DataFrame({"Protein Disease Linked"    : ["a", "c"], 
                                  "Protein Not Disease Linked": ["b", "d"]},
                                  index = ["Protein Entangled", "Protein Not Entangled"])

# print the output
print ("This is our (blank) contingency table:\n")

# create a table from our contingency_table using matplotlib
plt.clf()
fig, ax    = plt.subplots(figsize = (5, 2))
ax.axis("tight")
ax.axis("off")
cell_text  = contingency_table.reset_index().values.tolist()
col_labels = [""] + contingency_table.columns.tolist()
table      = ax.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(14)  
table.scale(2.5, 2)  
plt.show()

* As in **Example 1.1**, we now need to compute the values of (a, b, c, d), insert them into a DataFrame, and run the `fisher_exact` function from `scipy.stats` to compute the odds ratio and *p*-value

In [ ]:
# compute the values of {a, b, c, d} and construct the contingency table
a = len(data2[(data2["entanglement"] == "Yes" ) & (data2["disease-linked"] == "Yes")])
b = len(data2[(data2["entanglement"] == "Yes" ) & (data2["disease-linked"] == "No")])
c = len(data2[(data2["entanglement"] == "No"  ) & (data2["disease-linked"] == "Yes")])
d = len(data2[(data2["entanglement"] == "No"  ) & (data2["disease-linked"] == "No")])

# create the contingency table as a pandas DataFrame object
contingency_table = pd.DataFrame({"Protein Disease Linked"    : [a, c], 
                                  "Protein Not Disease Linked": [b, d]},
                                 index = ["Protein Entangled", "Protein Not Entangled"])

# print the contingency table
print ("This is our contingency table:\n")

# create a table from our contingency_table using matplotlib
plt.clf()
fig, ax    = plt.subplots(figsize = (5, 2))
ax.axis("tight")
ax.axis("off")
cell_text  = contingency_table.reset_index().values.tolist()
col_labels = [""] + contingency_table.columns.tolist()
table      = ax.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(14)  
table.scale(2.5, 2)  
plt.show()

# use the fisher_exact function from scipy.stats to compute the odds ratio and p-value
odds_ratio, fisher_p_value = fisher_exact(contingency_table, alternative = 'two-sided')

print ("The odds ratio is:", '%.2f' %odds_ratio)
print ("The p-value is   :", '%.2e' %fisher_p_value)

### Step 4 - Interpret the results

Which of the following is a correct interpretation of the results from our analysis?

* Entangled proteins are positively associated with disease
* Entangled proteins are negatively associated with disease
* There is no association between entangled proteins and disease

<details>
  <summary> Answer! (Click to expand)</summary>
  XXX
</details>

---

## Example 1.3

**Application 1.3**: We hypothesize that misfolding occurs preferentially in the entangled regions of a protein's primary structure

* This hypothesis is related to the hypothesis stated in **Application 1.1**; in this case, rather than considering whether each protein is entangled, we consider here individual amino acids and whether each is involved in an entanglement (detected by analysis of protein structures) and is misfolded (again from LiP-MS experiments).

* Take a moment to think - **what should your contingency table look like?** What are the columns, and what are the rows?

* Carry out this last example solo by running each of the following cells in sequence
* Once you have obtained the result at the end, try explaining your conclusions to at least one person sitting near you

### Step 0 - Load libraries

* We first need to make sure we have access to all of the functions etc. that we need for this analysis - let's load some libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

### Step 1 - Load the data
* We will use a data set containing per-residue information on entanglements and misfolding from LiP-MS

In [ ]:
# data3 is a pandas DataFrame object
data_path = "/home/jovyan/data-store/home/shared/NCEMS/2025-NCEMS-ML-webinar/"
data3     = pd.read_csv(data_path+"Ecoli_entanglement_data.csv")

### Step 2 - Explore the data
* Run a few simple commands to learn more about the data set

In [ ]:
# first, print a quick summary
print ("Create a quick summary of the DataFrame:\n")
data3.info()

# second, print the first 10 rows of the DataFrame
print ("\nPrint the first 10 rows of the DataFrame:\n")
data3.head(10)

* Consider the output from these commands - what can you learn about the data set we are using? (Consider, for example, the line in the output of `data3.info()` for `mapped_resid`).

* There are many columns in this file, but we only need a few: (1) `region` indicates whether an entanglement is present (`True`) or absent (`False`) at a residue; (2) `cut_C_Rall` indicates whether misfolding is detected at this residue by LiP-MS in the absence of chaperones.

* `cut_CD_Rall` & `cut_CG_Rall` indicate whether misfolding is detected at a residue in the presence of the molecular chaperones DnaK/J & GroEL/ES, respectively. We will only consider the chaperone-free case here.

* For these data, we need to do two additional processing steps to (1) remove rows of the DataFrame `data3` that are associated with proteins that had either low coverage or low abundance in the LiP-MS experiment and (2) remove rows of `data3` that correspond to amino acids present in the protein crystal structure but not in the UniProt sequence of the protein:

In [ ]:
# filter to remove proteins that have low abundance or coverage in LiP-MS experiments

# load a data set that contains a list of the high-quality proteins from LiP-MS
data3_mask = pd.read_csv(data_path+"Ecoli_C_high-abundance_high-coverage.txt")

# keep only entries that correspond to high-quality proteins from LiP-MS
data3_filtered = data3[data3['gene'].isin(data3_mask['gene'])]

# remove rows in which mapped_resid is not an integer value
data3_filtered = data3_filtered.dropna(subset = ['mapped_resid'])

# print a quick summary of the new data3_filtered DataFrame
print ("\nCreate a quick summary of the filtered DataFrame:\n")
data3_filtered.info()

* Comparing the info printed by `.info()` for `data3` and `data3_filtered`, we can see that the number of rows is reduced from 384,582 to 98,129
* There are now no missing values in any row

### Step 3 - Run the analysis

* Before running the full analysis below, compare your prediction for the contingency table's format to the table generated by the next cell. Were you correct?

In [ ]:
# print a blank contingency table in the format needed for this hypothesis
contingency_table = pd.DataFrame({"Residue Misfolded"    : ["a", "c"], 
                                  "Residue Not Misfolded": ["b", "d"]},
                                 index = ["Residue Entangled", "Residue Not Entangled"])

# print the output
print ("This is our (blank) contingency table:\n")

# create a table from our contingency_table using matplotlib
plt.clf()
fig, ax    = plt.subplots(figsize = (5, 2))
ax.axis("tight")
ax.axis("off")
cell_text  = contingency_table.reset_index().values.tolist()
col_labels = [""] + contingency_table.columns.tolist()
table      = ax.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(14)  
table.scale(2, 2)  
plt.show()

* This table is identical in format to the one employed in **Application 1.1** except "protein" is replaced by "residue"
* Run the cell below to complete the analysis

In [ ]:
# compute the values of {a, b, c, d} and construct the contingency table
a = len(data3_filtered[(data3_filtered["region"] == True ) & (data3_filtered["cut_C_Rall"] == True )])
b = len(data3_filtered[(data3_filtered["region"] == True ) & (data3_filtered["cut_C_Rall"] == False)])
c = len(data3_filtered[(data3_filtered["region"] == False) & (data3_filtered["cut_C_Rall"] == True )])
d = len(data3_filtered[(data3_filtered["region"] == False) & (data3_filtered["cut_C_Rall"] == False)])

# also, put values into a new format to enable a nice print statement
contingency_table = pd.DataFrame({"Residue Misfolded"    : [a, c], 
                                  "Residue Not Misfolded": [b, d]},
                                 index = ["Residue Entangled", "Residue Not Entangled"])

# print the contingency table
print ("This is our contingency table:\n")

# create a table from our contingency_table using matplotlib
plt.clf()
fig, ax    = plt.subplots(figsize = (5, 2))
ax.axis("tight")
ax.axis("off")
cell_text  = contingency_table.reset_index().values.tolist()
col_labels = [""] + contingency_table.columns.tolist()
table      = ax.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(14)  
table.scale(2, 2)  
plt.show()

# use the fisher_exact function from scipy.stats to compute the odds ratio and p-value
odds_ratio, fisher_p_value = fisher_exact(contingency_table, alternative = 'two-sided')

print ("The odds ratio is:", '%.2f' %odds_ratio)
print ("The p-value is   :", '%.2e' %fisher_p_value)

### Step 4 - Interpret the results

* Think about how you can state this result in simple language and then try to describe it to someone sitting near you
* Your explanation should include: (1) A conclusion about the association (*i.e.*, is there positive, negative, or no association) & (2) A statement about the significance of the result based on the computed *p*-value